# ANA500 Micro-Project 1 - Adult Income Dataset
Jesus Adrian Alvarado Gadea
ANA500, National University
This micro-project uses NumPy and Pandas to acquire and prepare the Adult Income dataset. Per the assignment, it stops at Step 2 (Prepare). No models are built here.
Problem statement: Earnings in the 1994 US workforce were spread unevenly across demographic and employment characteristics. People working similar hours ended up at very different income levels. I want to describe which recorded attributes are associated with earning more than $50,000 a year.
Hypothesis: Education and hours worked per week are the strongest correlates of earning above $50,000, with occupation and marital status adding some additional signal. This is stated here but not tested, since the project ends at Prepare.

In [1]:
# NumPy and Pandas only for this project, no modeling libraries
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 95)
pd.set_option("display.max_columns", 50)

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.2 | numpy 2.4.4


## Step 1 - Acquire
Source: Adult (Census Income) from the UCI Machine Learning Repository, dataset ID 2.
Citation: Becker, B. and Kohavi, R. (1996). Adult [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20. Licensed CC BY 4.0.
The data was pulled by Barry Becker from the 1994 US Census, filtered to age over 16, AGI over 100, weight over 1, and hours worked over 0. It has 48,842 rows, 14 predictors and a binary income target.
The repository splits it into adult.data (32,561 rows) and adult.test (16,281 rows). They are a train/test split of the same extract, not different populations, so I combine them and leave the splitting for later. There is no header row, missing values are stored as "?", and adult.test has a junk first line plus a period on every label.

In [2]:
# the raw files have no header row, so the names come from the UCI documentation
cols = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
        "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
        "hours_per_week", "native_country", "income"]

uci = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/"
mirror = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"
local = Path("data/adult-all.csv")

# skipinitialspace drops the space after every comma, na_values catches the "?" markers
def read(src, skip=0):
    return pd.read_csv(src, header=None, names=cols, skiprows=skip,
                       skipinitialspace=True, na_values=["?"])

# try the repository first, fall back to the cached copy if there is no connection
try:
    train = read(uci + "adult.data")
    test = read(uci + "adult.test", skip=1)
    df_raw = pd.concat([train, test], ignore_index=True)
    source = "UCI repository"
except Exception:
    df_raw = read(local) if local.exists() else read(mirror)
    source = "local cache" if local.exists() else "mirror"

print("loaded from:", source)
print(df_raw.shape)

loaded from: local cache
(48842, 15)


In [3]:
# quick look to confirm the column names line up with the values
df_raw.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
# data types and non-null counts
df_raw.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       46043 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education_num   48842 non-null  int64
 5   marital_status  48842 non-null  str  
 6   occupation      46033 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital_gain    48842 non-null  int64
 11  capital_loss    48842 non-null  int64
 12  hours_per_week  48842 non-null  int64
 13  native_country  47985 non-null  str  
 14  income          48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 26.4 MB


In [5]:
# baseline numbers to compare against later
start_rows = len(df_raw)
start_cols = df_raw.shape[1]
start_mem = df_raw.memory_usage(deep=True).sum() / 1024**2
start_missing = int(df_raw.isna().sum().sum())

print("rows", start_rows)
print("cols", start_cols)
print("memory MB", round(start_mem, 2))
print("missing cells", start_missing)
print("duplicate rows", int(df_raw.duplicated().sum()))

rows 48842
cols 15
memory MB 26.36
missing cells 6465
duplicate rows 52


Data dictionary:
- age - age in years
- workclass - employer type
- fnlwgt - Census final weight (a sampling weight, not a personal attribute)
- education - highest level as text
- education_num - highest level as a number
- marital_status - marital status
- occupation - occupation category
- relationship - role in the household
- race - self reported race
- sex - self reported sex
- capital_gain - capital gains in dollars, top coded at 99999
- capital_loss - capital losses in dollars
- hours_per_week - usual hours worked
- native_country - country of origin
- income - target, above or below 50K
The fnlwgt and education entries matter later. fnlwgt describes the survey design rather than the person, and education looks like a duplicate of education_num.

## Step 2 - Prepare

## Cleaning up formatting first
Stripping whitespace, catching any leftover "?" values, and fixing the target labels before I measure anything, so the numbers aren't thrown off by formatting.

In [6]:
# keep the raw version so I can compare against it at the end
df = df_raw.copy()

# strip spaces and catch any '?' the parser missed
text_cols = df.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    df[c] = df[c].str.strip()
    df[c] = pd.Series(np.where(df[c].isin(["?", ""]), np.nan, df[c]), index=df.index)

# adult.test puts a period on every label, which gives 4 target values instead of 2
df["income"] = df["income"].str.rstrip(".").str.strip()

# should be exactly two values now

print(sorted(df["income"].dropna().unique()))
print(df["income"].value_counts())
print((df["income"].value_counts(normalize=True) * 100).round(2))

['<=50K', '>50K']
income
<=50K    37155
>50K     11687
Name: count, dtype: int64
income
<=50K    76.07
>50K     23.93
Name: proportion, dtype: float64


The target is imbalanced, about 76% to 24%. I am noting that but not fixing it here. Resampling belongs with the modeling step, not with preparation.

## Duplicates

In [7]:
# rows that repeat across every single column
n_dupes = int(df.duplicated().sum())
print("exact duplicate rows:", n_dupes)

df = df.drop_duplicates().reset_index(drop=True)  # keeps the first of each
print("rows now:", len(df))

exact duplicate rows: 52
rows now: 48790


## Missing values
Three columns have missing values. Before deciding what to do about them I checked two things: whether they follow a pattern, and whether they relate to the target.

In [8]:
# how much is missing and where
missing = df.isna().sum()
have_na = missing[missing > 0]
print(pd.DataFrame({"n": have_na, "pct": (have_na / len(df) * 100).round(2)}))

# how many rows would listwise deletion actually cost me
any_na = df.isna().any(axis=1)
print("\nrows with any missing:", int(any_na.sum()), round(any_na.mean() * 100, 2), "%")

                   n   pct
workclass       2795  5.73
occupation      2805  5.75
native_country   856  1.75

rows with any missing: 3615 7.41 %


In [9]:
# check 1: do the missing values follow a pattern?
na_work = df["workclass"].isna()
na_occ = df["occupation"].isna()

print("workclass missing:", int(na_work.sum()))
print("occupation missing:", int(na_occ.sum()))
print("both missing:", int((na_work & na_occ).sum()))
print("occupation only:", int((na_occ & ~na_work).sum()))
print("workclass only:", int((na_work & ~na_occ).sum()))
print()
print(df.loc[na_occ & ~na_work, "workclass"].value_counts())

workclass missing: 2795
occupation missing: 2805
both missing: 2795
occupation only: 10
workclass only: 0

workclass
Never-worked    10
Name: count, dtype: int64


So the missing values are not random. Every row missing workclass is also missing occupation, and the only rows missing occupation on its own are the ones where workclass is "Never-worked". Someone who has never worked would not have an occupation, so these blanks are recording a real situation.

In [10]:
# check 2: are the rows with missing values different from the rest?
# compare the target rate between incomplete and complete rows
above = df["income"].eq(">50K")
print("rate above 50K, rows with missing values:", round(above[any_na].mean() * 100, 2))
print("rate above 50K, complete rows:", round(above[~any_na].mean() * 100, 2))

rate above 50K, rows with missing values: 13.25
rate above 50K, complete rows: 24.8


That is a big gap, roughly 13% against 25%. If I dropped the incomplete rows I would be throwing out a group that earns much less on average, which would push the overall rate up and distort the thing I am trying to describe.
So instead of dropping or imputing them, I am filling them with an "Unknown" category. That keeps every row, and it lets a later model use the missingness itself since it clearly carries information. Imputing with the mode would be worse because it would claim a specific job and employer for people who are not in the workforce at all.

In [11]:
# fill with a category instead of dropping the rows
for c in ["workclass", "occupation", "native_country"]:
    df[c] = pd.Series(np.where(df[c].isna(), "Unknown", df[c]), index=df.index)

print("missing cells left:", int(df.isna().sum().sum()))
print(df["workclass"].value_counts())

missing cells left: 0
workclass
Private             33860
Self-emp-not-inc     3861
Local-gov            3136
Unknown              2795
State-gov            1981
Self-emp-inc         1694
Federal-gov          1432
Without-pay            21
Never-worked           10
Name: count, dtype: int64


## education and education_num
These look like the same variable stored twice, so I checked instead of assuming.

In [12]:
# if each maps to exactly one of the other, they are the same variable
per_code = df.groupby("education_num")["education"].nunique()
per_label = df.groupby("education")["education_num"].nunique()

print("max labels per code:", per_code.max())
print("max codes per label:", per_label.max())

mapping = (df[["education_num", "education"]]
           .drop_duplicates().sort_values("education_num"))
print(mapping.to_string(index=False))

max labels per code: 1
max codes per label: 1
 education_num    education
             1    Preschool
             2      1st-4th
             3      5th-6th
             4      7th-8th
             5          9th
             6         10th
             7         11th
             8         12th
             9      HS-grad
            10 Some-college
            11    Assoc-voc
            12   Assoc-acdm
            13    Bachelors
            14      Masters
            15  Prof-school
            16    Doctorate


It is a one to one match, and the numbers run in the right order from Preschool at 1 up to Doctorate at 16. I am keeping education_num because it is already an ordered numeric scale, and dropping the text version. I saved the mapping first so the labels are still available if I need to report results in words.

In [13]:
# save the labels before dropping the text column so I can still report in words
edu_lookup = dict(zip(mapping["education_num"], mapping["education"]))
df = df.drop(columns=["education"])
print("columns:", df.shape[1])

columns: 14


## fnlwgt
This is the Census final weight. It estimates how many people in the population each row stands for, so it describes the survey design and not the person. Using it as a predictor is a common mistake with this dataset. If it really did describe the individual I would expect it to correlate with something personal, so I checked.

In [14]:
# fnlwgt spans a huge range, which already looks more like a weight than an attribute
print(df["fnlwgt"].describe())

# a real personal attribute should correlate with something
num_cols = ["fnlwgt", "age", "education_num", "hours_per_week",
            "capital_gain", "capital_loss"]
corr = df[num_cols].corr()["fnlwgt"].drop("fnlwgt")
print()
print(corr.round(4))
print("\nlargest absolute correlation:", round(corr.abs().max(), 4))

count    4.879000e+04
mean     1.896690e+05
std      1.056172e+05
min      1.228500e+04
25%      1.175550e+05
50%      1.781385e+05
75%      2.376062e+05
max      1.490400e+06
Name: fnlwgt, dtype: float64

age              -0.0765
education_num    -0.0387
hours_per_week   -0.0135
capital_gain     -0.0037
capital_loss     -0.0044
Name: fnlwgt, dtype: float64

largest absolute correlation: 0.0765


In [15]:
# still available in df_raw if I ever need weighted population estimates
df = df.drop(columns=["fnlwgt"])
print("columns:", df.shape[1])

columns: 13


Correlations are all near zero, which fits a design weight. Dropped it from the working data, though it is still in df_raw if I ever need weighted population estimates.
One side effect worth checking: dropping columns can turn rows that used to be different into identical rows.

In [16]:
# did dropping columns create new duplicates?
print("rows identical to another row now:", int(df.duplicated().sum()))
print("as a percentage:", round(df.duplicated().mean() * 100, 2))

# group on every column to see how often each attribute profile repeats
profiles = df.groupby(list(df.columns), dropna=False, observed=True).size()
print("distinct profiles:", len(profiles))
print("profiles appearing more than once:", int((profiles > 1).sum()))
print("most repeats of one profile:", int(profiles.max()))
print()
# as a frame so the wide index wraps instead of running off the page
print(profiles.sort_values(ascending=False).head(3).reset_index(name="count"))

rows identical to another row now: 6322


as a percentage: 12.96


distinct profiles: 42468
profiles appearing more than once: 3451
most repeats of one profile: 21



   age workclass  education_num      marital_status    occupation relationship   race   sex  \
0   33   Private              9  Married-civ-spouse  Craft-repair      Husband  White  Male   
1   32   Private              9  Married-civ-spouse  Craft-repair      Husband  White  Male   
2   35   Private              9  Married-civ-spouse  Craft-repair      Husband  White  Male   

   capital_gain  capital_loss  hours_per_week native_country income  count  
0             0             0              40  United-States  <=50K     21  
1             0             0              40  United-States  <=50K     20  
2             0             0              40  United-States  <=50K     20  


About 13% of rows now match another row, and the most common profile shows up 21 times: a 33 year old married man working 40 hours in private sector craft repair. That is not an error, it is just a very common profile in the 1994 workforce.
I am keeping these. The 52 rows I removed earlier were identical across all 15 columns including the survey weight, and two separate people getting the exact same weight by chance is very unlikely, so those looked like genuine duplicates. These new ones only match on the attributes I kept, so they are different people who happen to look the same on paper. Deleting them would make common profiles look rare.

## Outliers

In [17]:
# standard IQR outlier check
for c in ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss"]:
    v = df[c].to_numpy()
    q1, q3 = np.percentile(v, [25, 75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = int(np.sum((v < low) | (v > high)))
    pct = round(flagged / len(v) * 100, 2)
    print(f"{c}: min {v.min()} max {v.max()} "
          f"bounds ({low}, {high}) flagged {flagged} ({pct}%)")

age: min 17 max 90 bounds (-2.0, 78.0) flagged 215 (0.44%)
education_num: min 1 max 16 bounds (4.5, 16.5) flagged 1787 (3.66%)
hours_per_week: min 1 max 99 bounds (32.5, 52.5) flagged 13486 (27.64%)
capital_gain: min 0 max 99999 bounds (0.0, 0.0) flagged 4035 (8.27%)
capital_loss: min 0 max 4356 bounds (0.0, 0.0) flagged 2282 (4.68%)


In [18]:
# checking why the IQR flagged so many rows
print(df["hours_per_week"].value_counts().head(6))
pct40 = (df["hours_per_week"] == 40).mean() * 100
print("\npct working exactly 40 hours:", round(pct40, 2))

hours_per_week
40    22773
50     4242
45     2715
60     2177
35     1934
20     1860
Name: count, dtype: int64

pct working exactly 40 hours: 46.68


The IQR rule flags almost 28% of hours_per_week, which is clearly too many. The reason is that nearly half the sample works exactly 40 hours, so the interquartile range collapses to 5 hours and the upper cutoff lands at 52.5. That would make a 55 hour work week an outlier, which does not make sense.
So I am not deleting anything based on the IQR. I checked the ranges against what is actually possible instead: age runs 17 to 90 which matches the documented filter, hours run 1 to 99 which are all achievable, and education_num covers the full 1 to 16 scale. Every value in the data is plausible.

## Capital gain and loss

In [19]:
# both columns are mostly zeros, so the plain mean will be misleading
for c in ["capital_gain", "capital_loss"]:
    v = df[c].to_numpy()
    print(c, "| pct zero:", round(np.mean(v == 0) * 100, 2),
          "| max:", v.max(),
          "| mean when not zero:", round(v[v > 0].mean(), 2))

print("\nrows at exactly 99999:", int((df["capital_gain"] == 99999).sum()))
print(df.loc[df["capital_gain"] > 0, "capital_gain"].value_counts().head(6))

capital_gain

 | pct zero: 91.73 | max: 99999 | mean when not zero: 13061.67
capital_loss | pct zero: 95.32 | max: 4356 | mean when not zero: 1872.83

rows at exactly 99999: 244
capital_gain
15024    513
7688     410
7298     364
99999    244
3103     152
5178     146
Name: count, dtype: int64


Both columns are mostly zeros, so the plain average of either one describes almost nobody. The value 99999 shows up far more often than the values around it, which is a top code - the Census caps the number to protect people's privacy. Those rows mean "at least 99999", not exactly that amount.
I am not removing them, since they are the highest earners and that is exactly the group the hypothesis is about. I flag them with an indicator column further down instead.

## Grouping small categories

In [20]:
# group countries with too few rows to be useful
counts = df["native_country"].value_counts()
rare = counts[counts < 100].index  # 100 rows is my cutoff for keeping a level separate

print("levels before:", df["native_country"].nunique())
print("levels under 100 rows:", len(rare),
      "covering", int(counts[rare].sum()), "records")

df["native_country"] = pd.Series(
    np.where(df["native_country"].isin(rare), "Other", df["native_country"]),
    index=df.index)

print("levels after:", df["native_country"].nunique())

levels before: 42
levels under 100 rows: 26 covering 1211 records
levels after: 17


In [21]:
# Without-pay and Never-worked are tiny and mean roughly the same thing
df["workclass"] = pd.Series(
    np.where(df["workclass"].isin(["Without-pay", "Never-worked"]),
             "No-pay-or-never-worked", df["workclass"]),
    index=df.index)

print(df["workclass"].value_counts())

workclass
Private                   33860
Self-emp-not-inc           3861
Local-gov                  3136
Unknown                    2795
State-gov                  1981
Self-emp-inc               1694
Federal-gov                1432
No-pay-or-never-worked       31
Name: count, dtype: int64

In [22]:
# check how much these two columns say the same thing
overlap = pd.crosstab(df["relationship"], df["marital_status"])
print(overlap)
print()
print((overlap.max(axis=1) / overlap.sum(axis=1) * 100).round(2))

marital_status  Divorced  Married-AF-spouse  Married-civ-spouse  Married-spouse-absent  \
relationship                                                                             
Husband                0                 12               19691                      0   
Not-in-family       3626                  0                  23                    329   
Other-relative       181                  1                 201                     54   
Own-child            455                  1                 143                     61   
Unmarried           2368                  0                   0                    183   
Wife                   0                 23                2308                      0   

marital_status  Never-married  Separated  Widowed  
relationship                                       
Husband                     0          0        0  
Not-in-family            7091        637      851  
Other-relative            920         79       70  
Own-child          

relationship and marital_status overlap a lot - Husband is almost always Married-civ-spouse. But they are not the same, because Not-in-family, Own-child and Unmarried each spread across several marital statuses and say something about the household that marital_status alone does not. I am keeping both. Overlapping predictors are a modeling problem, and dropping a column now to solve a problem I have not reached yet would be premature.

## New columns

In [23]:
# flags for the target and for the capital columns
df["income_gt_50k"] = np.where(df["income"] == ">50K", 1, 0).astype("int8")
df["capital_gain_topcoded"] = np.where(df["capital_gain"] == 99999, 1, 0).astype("int8")
df["has_capital_gain"] = np.where(df["capital_gain"] > 0, 1, 0).astype("int8")
df["has_capital_loss"] = np.where(df["capital_loss"] > 0, 1, 0).astype("int8")
# one signed column instead of two
df["net_capital"] = df["capital_gain"] - df["capital_loss"]

# log1p works on zeros, which matters here since the column is over 90% zeros
df["capital_gain_log"] = np.log1p(df["capital_gain"])  # log(1+x), safe at zero

print("skew before:", round(df["capital_gain"].skew(), 3))
print("skew after:", round(df["capital_gain_log"].skew(), 3))

skew before: 11.888
skew after: 3.111


In [24]:
# bucket the continuous columns into readable groups
edu = df["education_num"].to_numpy()
df["education_tier"] = np.select(
    [edu <= 8, edu == 9, (edu >= 10) & (edu <= 12), edu >= 13],
    ["No-HS-diploma", "HS-graduate", "Some-college", "Bachelors-plus"],
    default="Unknown")

hrs = df["hours_per_week"].to_numpy()
df["hours_band"] = np.select(
    [hrs < 35, (hrs >= 35) & (hrs <= 40), (hrs > 40) & (hrs <= 50), hrs > 50],
    ["Part-time", "Full-time", "Overtime", "Long-hours"],
    default="Unknown")

# digitize returns the bin index, which I use to pick the label
age_labels = np.array(["17-24", "25-34", "35-44", "45-54", "55-64", "65-plus"])
age_edges = np.array([25, 35, 45, 55, 65])
df["age_band"] = age_labels[np.digitize(df["age"].to_numpy(), age_edges)]

for c in ["education_tier", "hours_band", "age_band"]:
    print(df[c].value_counts().sort_index(), "\n")

education_tier
Bachelors-plus    12097
HS-graduate       15770
No-HS-diploma      6399
Some-college      14524
Name: count, dtype: int64 



hours_band
Full-time     26061
Long-hours     5434
Overtime       8909
Part-time      8386
Name: count, dtype: int64 

age_band
17-24       8409
25-34      12564
35-44      12184
45-54       8765
55-64       4782
65-plus     2086
Name: count, dtype: int64 



In [25]:
# nothing should have fallen through to the default
for c in ["education_tier", "hours_band"]:
    print(c, "rows set to Unknown:", int((df[c] == "Unknown").sum()))

education_tier rows set to Unknown: 0
hours_band rows set to Unknown: 0


## Data types
Converting the text columns to category and shrinking the integers, which cuts the memory use a lot.

In [26]:
# category dtype stores each level once instead of per row
for c in ["workclass", "marital_status", "occupation", "relationship", "race", "sex",
          "native_country", "income", "education_tier", "hours_band", "age_band"]:
    df[c] = df[c].astype("category")

# int8 holds anything up to 127, which covers all three of these
for c in ["age", "education_num", "hours_per_week"]:
    df[c] = df[c].astype("int8")
for c in ["capital_gain", "capital_loss", "net_capital"]:
    df[c] = df[c].astype("int32")

df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 48790 entries, 0 to 48789
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   age                    48790 non-null  int8    
 1   workclass              48790 non-null  category
 2   education_num          48790 non-null  int8    
 3   marital_status         48790 non-null  category
 4   occupation             48790 non-null  category
 5   relationship           48790 non-null  category
 6   race                   48790 non-null  category
 7   sex                    48790 non-null  category
 8   capital_gain           48790 non-null  int32   
 9   capital_loss           48790 non-null  int32   
 10  hours_per_week         48790 non-null  int8    
 11  native_country         48790 non-null  category
 12  income                 48790 non-null  category
 13  income_gt_50k          48790 non-null  int8    
 14  capital_gain_topcoded  48790 non-null  int8    
 

## Checks
Running through everything the cleaning was supposed to accomplish.

In [27]:
# everything the cleaning was meant to do
checks = {
    "no missing values": int(df.isna().sum().sum()) == 0,
    "rows match after dedupe": len(df) == start_rows - n_dupes,
    "target is 0/1": set(df["income_gt_50k"].unique()) == {0, 1},
    "target matches label": bool(
        (df["income"].eq(">50K").astype(int) == df["income_gt_50k"]).all()),
    "education and fnlwgt gone": ("education" not in df.columns
                                  and "fnlwgt" not in df.columns),
    "age 17 to 90": bool(df["age"].between(17, 90).all()),
    "hours 1 to 99": bool(df["hours_per_week"].between(1, 99).all()),
    "education_num 1 to 16": bool(df["education_num"].between(1, 16).all()),
    "no Unknown tiers": int((df["education_tier"] == "Unknown").sum()) == 0,
}

for k, v in checks.items():
    print(("pass" if v else "FAIL"), "-", k)

assert all(checks.values())

pass - no missing values
pass - rows match after dedupe
pass - target is 0/1
pass - target matches label
pass - education and fnlwgt gone
pass - age 17 to 90
pass - hours 1 to 99
pass - education_num 1 to 16
pass - no Unknown tiers


In [28]:
# before and after comparison
end_mem = df.memory_usage(deep=True).sum() / 1024**2

print("rows:", start_rows, "->", len(df))
print("cols:", start_cols, "->", df.shape[1])
print("missing cells:", start_missing, "-> 0")
print("memory MB:", round(start_mem, 2), "->", round(end_mem, 2))
print("kept", round(len(df) / start_rows * 100, 2), "% of rows")

rows: 48842 -> 48790
cols: 15 -> 22
missing cells: 6465 -> 0
memory MB: 26.36 -> 1.77
kept 99.89 % of rows


In [29]:
# make sure the cleaning did not shift the thing I am studying
raw_income = df_raw["income"].str.strip().str.rstrip(".")
raw_rate = raw_income.eq(">50K").mean() * 100
clean_rate = df["income_gt_50k"].mean() * 100

print("above 50K, raw:", round(raw_rate, 2))
print("above 50K, cleaned:", round(clean_rate, 2))
print("difference:", round(clean_rate - raw_rate, 2), "percentage points")

above 50K, raw: 23.93
above 50K, cleaned: 23.94
difference: 0.01 percentage points


In [30]:
# final structure
print(df.shape)
print(pd.DataFrame({"dtype": df.dtypes.astype(str), "unique": df.nunique()}))

(48790, 22)
                          dtype  unique
age                        int8      74
workclass              category       8
education_num              int8      16
marital_status         category       7
occupation             category      15
relationship           category       6
race                   category       5
sex                    category       2
capital_gain              int32     123
capital_loss              int32      99
hours_per_week             int8      96
native_country         category      17
income                 category       2
income_gt_50k              int8       2
capital_gain_topcoded      int8       2
has_capital_gain           int8       2
has_capital_loss           int8       2
net_capital               int32     221
capital_gain_log        float64     123
education_tier         category       4
hours_band             category       4
age_band               category       6


In [31]:
# save for Micro-Projects 2 through 4
out = Path("data/adult_income_prepared.csv")
df.to_csv(out, index=False)

# reload to confirm the file is readable and complete
check = pd.read_csv(out)
print("saved to", out)
print("reloaded shape:", check.shape, "| matches:", check.shape == df.shape)

saved to data/adult_income_prepared.csv
reloaded shape: (48790, 22) | matches: True


## Where this stops
The assignment has Micro-Project 1 ending at Step 2, so Analyze, Report and Act are not done here. Analyze and Act come in Micro-Project 3, and Report comes in Micro-Project 2. The hypothesis from the top is still untested on purpose.
What I ended up doing and why:
- Missing values became an "Unknown" category rather than being dropped, because they follow
  a clear pattern and the rows with them earn much less than average
- Removed 52 exact duplicate rows, but kept the rows that only became identical after I
  dropped columns
- Dropped education since it duplicates education_num, and dropped fnlwgt since it is a
  survey weight rather than something about the person
- Left the 99999 capital gain values alone since they are top coded, not errors
- Did not remove anything for being an outlier, since the IQR result was misleading
- Grouped native_country levels under 100 rows into Other, and merged two tiny workclass
  levels
- Left the class imbalance alone since fixing it belongs with modeling
Limitations: this is 1994 Census data, so it describes that year and not the labor market now. The dollar amounts, job categories and workforce makeup have all changed. The data also records race, sex and native country. I kept those columns because dropping them would hide differences rather than remove them, and any model built on this file should be checked for whether it performs differently across those groups. It is also observational data, so anything found in it is an association and not a cause.